# Heading

In [ ]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import os
import re
import nltk
from nltk.corpus import stopwords
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import pickle
import PyPDF2

In [ ]:
try:
    STOPWORDS = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    STOPWORDS = set(stopwords.words('english'))

In [ ]:
def advanced_clean_text(text):
    if not isinstance(text, str) or text.lower() == "nan" or text.strip() == "":
        return ""
    text = text.lower()
    text = re.sub(r"[\[\]'\".;:\|%()]", " ", text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = [word for word in text.split() if word not in STOPWORDS]
    return " ".join(tokens)

In [ ]:
def export_processed_resume_data(input_csv, output_csv):
    if not os.path.exists(input_csv):
        print(f"Error: Dataset not found at {input_csv}")
        return
    print("Step 1: Reading raw dataset...")
    df = pd.read_csv(input_csv)
    jd_cols = [
        '﻿job_position_name',
        'educationaL_requirements',
        'experiencere_requirement',
        'age_requirement',
        'responsibilities.1',
        'skills_required'
    ]
    resume_cols = [
        'address',
        'career_objective',
        'skills',
        'educational_institution_name',
        'degree_names',
        'passing_years',
        'educational_results',
        'result_types',
        'major_field_of_studies'
    ]
    all_target_cols = jd_cols + resume_cols + ['matched_score']
    print("Step 2: Filtering columns and cleaning text...")
    df_export = df[all_target_cols].copy()
    for col in jd_cols + resume_cols:
        df_export[col] = df_export[col].astype(str).apply(advanced_clean_text)
    print("Step 3: Calculating ATS scores...")
    df_export['ats_score'] = df_export['matched_score'] * 100
    df_export = df_export.drop(columns=['matched_score'])
    print(f"Step 4: Exporting final data to {output_csv}...")
    df_export.to_csv(output_csv, index=False)
    print("\n" + "="*30)
    print(" DATA EXPORT SUCCESSFUL")
    print(f"Total Records: {len(df_export)}")
    print(f"File Path: {output_csv}")
    print("="*30)

In [ ]:
INPUT_FILE = '/content/resume_data.csv'   ###https://www.kaggle.com/datasets/saugataroyarghya/resume-dataset/data
OUTPUT_FILE = '/content/final_processed_data.csv'
export_processed_resume_data(INPUT_FILE, OUTPUT_FILE)

Step 1: Reading raw dataset...
Step 2: Filtering columns and cleaning text...
Step 3: Calculating ATS scores...
Step 4: Exporting final data to /content/final_processed_data.csv...

 DATA EXPORT SUCCESSFUL
Total Records: 9544
File Path: /content/final_processed_data.csv


# --- 1. THE DATASET CLASS WITH ATTENTION MASK LOGIC ---


In [ ]:
class SiameseResumeDataset(Dataset):
    def __init__(self, resumes, jobs, labels, max_len=256):
        self.resumes = resumes
        self.jobs = jobs
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def create_mask(self, sequence):
        return (sequence != 0).to(torch.float32)

    def __getitem__(self, idx):
        res_seq = torch.tensor(self.resumes[idx], dtype=torch.long)
        job_seq = torch.tensor(self.jobs[idx], dtype=torch.long)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        res_mask = self.create_mask(res_seq)
        job_mask = self.create_mask(job_seq)
        return {
            'resume': res_seq,
            'job': job_seq,
            'res_mask': res_mask,
            'job_mask': job_mask,
            'label': label
        }

In [ ]:
def build_vocab(texts, max_vocab_size=10000):
    """
    Creates a word-to-index mapping from the provided text data.
    0 is reserved for padding, and 1 is reserved for unknown words (OOV).
    """
    word_counts = {}
    for text in texts:
        for word in str(text).split():
            word_counts[word] = word_counts.get(word, 0) + 1

    # Sort words by frequency and take the top N
    sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)

    # Vocabulary starts with Padding and Unknown tokens
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for i, (word, _) in enumerate(sorted_words[:max_vocab_size]):
        vocab[word] = i + 2 # Shift by 2 to account for PAD and UNK

    print(f"✅ Vocabulary Built! Total Unique Tokens: {len(vocab)}")
    return vocab

def encode(text, vocab, max_len=256):
    """
    Converts a string of text into a list of integer IDs based on the vocab.
    Includes truncation and padding to ensure consistent sequence length.
    """
    tokens = str(text).split()
    # Map words to IDs, use 1 (<UNK>) if word not in vocab
    encoded = [vocab.get(word, 1) for word in tokens[:max_len]]

    # Pad with 0 (<PAD>) if the sequence is shorter than max_len
    padding_needed = max_len - len(encoded)
    encoded.extend([0] * padding_needed)

    return encoded

In [ ]:
def prepare_tensors_for_training(csv_path, vocab_path):
    if not os.path.exists(csv_path):
        print(f" Error: Processed CSV not found at {csv_path}. Please run Stage 1 Export first.")
        return None, None
    os.makedirs(os.path.dirname(vocab_path), exist_ok=True)
    print("Step 1: Loading cleaned dataset...")
    df = pd.read_csv(csv_path).fillna("")
    if os.path.exists(vocab_path):
        print(f" Loading existing vocabulary from {vocab_path}...")
        with open(vocab_path, "rb") as f:
            vocab = pickle.load(f)
    else:
        print(" Vocab file not found. Re-building vocabulary from CSV data...")
        all_text = (df['skills'] + " " + df['career_objective'] + " " +
                    df['job_position_name'] + " " + df['responsibilities.1']).astype(str)
        vocab = build_vocab(all_text)
        with open(vocab_path, "wb") as f:
            pickle.dump(vocab, f)
        print(" New vocabulary saved successfully.")
    print("Step 2: Encoding text to integer sequences...")
    res_text = (df['skills'] + " " + df['career_objective'] + " " + df['degree_names']).astype(str)
    job_text = (df['job_position_name'] + " " + df['responsibilities.1'] + " " + df['skills_required']).astype(str)
    res_enc = [encode(t, vocab) for t in res_text]
    job_enc = [encode(t, vocab) for t in job_text]
    labels = (df['ats_score'] / 100).tolist()
    full_dataset = SiameseResumeDataset(res_enc, job_enc, labels)
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_ds, val_ds = random_split(full_dataset, [train_size, val_size])
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32)
    print(f" Preprocessing Complete! Training batches: {len(train_loader)}")
    return train_loader, val_loader

# --- 3. UPDATED MODEL FOR MASKING ---


In [ ]:
class MaskedSiameseTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim=128):
        super(MaskedSiameseTransformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size + 1, embed_dim, padding_idx=0)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Sequential(
            nn.Linear(embed_dim * 3, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward_once(self, x, mask):
        x = self.embedding(x)
        bool_mask = (mask == 0)
        x = self.transformer(x, src_key_padding_mask=bool_mask)
        return torch.mean(x, dim=1)

    def forward(self, res, job, res_mask, job_mask):
        u = self.forward_once(res, res_mask)
        v = self.forward_once(job, job_mask)
        combined = torch.cat((u, v, torch.abs(u - v)), dim=1)
        return self.fc(combined).squeeze()

In [ ]:
def train_siamese_model():
    CSV_PATH = '/content/final_processed_data.csv'
    VOCAB_PATH = '/content/checkpoints/efficient_vocab.pkl'
    MODEL_SAVE_PATH = '/content/checkpoints/masked_siamese_model.pth'
    train_loader, val_loader = prepare_tensors_for_training(CSV_PATH, VOCAB_PATH)
    with open(VOCAB_PATH, "rb") as f:
        vocab = pickle.load(f)
    vocab_size = len(vocab)
    model = MaskedSiameseTransformer(vocab_size=vocab_size).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0005)
    best_val_loss = float('inf')
    epochs = 10
    print(f"\n Starting Training on {device}...")
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch in train_loader:
            res = batch['resume'].to(device)
            job = batch['job'].to(device)
            res_mask = batch['res_mask'].to(device)
            job_mask = batch['job_mask'].to(device)
            labels = batch['label'].to(device)
            optimizer.zero_grad()
            outputs = model(res, job, res_mask, job_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                res, job = batch['resume'].to(device), batch['job'].to(device)
                res_m, job_m = batch['res_mask'].to(device), batch['job_mask'].to(device)
                labels = batch['label'].to(device)

                outputs = model(res, job, res_m, job_m)
                val_loss += criterion(outputs, labels).item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print("  Model checkpoint saved!")

    print(f"\n Training complete. Final model saved at: {MODEL_SAVE_PATH}")


In [ ]:
# Check for GPU availability, otherwise fallback to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train_siamese_model()

Step 1: Loading cleaned dataset...
 Loading existing vocabulary from /content/checkpoints/efficient_vocab.pkl...
Step 2: Encoding text to integer sequences...
 Preprocessing Complete! Training batches: 239

 Starting Training on cuda...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch [1/10] | Train Loss: 0.6151 | Val Loss: 0.6643
  Model checkpoint saved!
Epoch [2/10] | Train Loss: 0.6038 | Val Loss: 0.6584
  Model checkpoint saved!
Epoch [3/10] | Train Loss: 0.6002 | Val Loss: 0.6574
  Model checkpoint saved!
Epoch [4/10] | Train Loss: 0.5982 | Val Loss: 0.6545
  Model checkpoint saved!
Epoch [5/10] | Train Loss: 0.5963 | Val Loss: 0.6574
Epoch [6/10] | Train Loss: 0.5948 | Val Loss: 0.6535
  Model checkpoint saved!
Epoch [7/10] | Train Loss: 0.5939 | Val Loss: 0.6557
Epoch [8/10] | Train Loss: 0.5929 | Val Loss: 0.6534
  Model checkpoint saved!
Epoch [9/10] | Train Loss: 0.5917 | Val Loss: 0.6520
  Model checkpoint saved!
Epoch [10/10] | Train Loss: 0.5903 | Val Loss: 0.6535

 Training complete. Final model saved at: /content/checkpoints/masked_siamese_model.pth


In [ ]:
def run_final_check(resume_pdf_path, jd_txt_path, model_path, vocab_path):
    if not all(os.path.exists(p) for p in [resume_pdf_path, jd_txt_path, model_path, vocab_path]):
        print(" Error: One or more files (PDF, TXT, Model, or Vocab) are missing.")
        return
    with open(vocab_path, "rb") as f:
        vocab = pickle.load(f)
    vocab_size = len(vocab)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MaskedSiameseTransformer(vocab_size=vocab_size).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print("Step 1: Extracting text from files...")
    resume_text = ""
    with open(resume_pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            resume_text += page.extract_text() + " "
    with open(jd_txt_path, "r", encoding="utf-8") as f:
        jd_text = f.read()
    clean_resume = advanced_clean_text(resume_text)
    clean_jd = advanced_clean_text(jd_text)
    print("Step 2: Preparing tensors...")
    res_seq = torch.tensor([encode(clean_resume, vocab)], dtype=torch.long).to(device)
    jd_seq = torch.tensor([encode(clean_jd, vocab)], dtype=torch.long).to(device)
    res_mask = (res_seq != 0).to(torch.float32).to(device)
    job_mask = (jd_seq != 0).to(torch.float32).to(device)
    print("Step 3: Calculating Match Score...")
    with torch.no_grad():
        logits = model(res_seq, jd_seq, res_mask, job_mask)
        probability = torch.sigmoid(logits).item()
        match_percentage = probability * 100
    print("\n" + "="*40)
    print(" FINAL ATS CHECK REPORT")
    print("="*40)
    print(f" Resume: {os.path.basename(resume_pdf_path)}")
    print(f" Job Description: {os.path.basename(jd_txt_path)}")
    print("-" * 40)
    print(f" PREDICTED MATCH SCORE: {match_percentage:.2f}%")
    print("="*40)
    if match_percentage > 75:
        print(" Result: Strong Candidate - Highly Recommended.")
    elif match_percentage > 50:
        print(" Result: Potential Match - Review manually.")
    else:
        print(" Result: Low Match - Profile lacks key requirements.")

In [ ]:
RESUME_FILE = "/content/sample_resume.pdf"
JD_FILE = "/content/jobdes.txt"
MODEL_WEIGHTS = "/content/checkpoints/masked_siamese_model.pth"
VOCAB_FILE = "/content/checkpoints/efficient_vocab.pkl"
run_final_check(RESUME_FILE, JD_FILE, MODEL_WEIGHTS, VOCAB_FILE)

Step 1: Extracting text from files...
Step 2: Preparing tensors...
Step 3: Calculating Match Score...

 FINAL ATS CHECK REPORT
 Resume: sample_resume.pdf
 Job Description: jobdes.txt
----------------------------------------
 PREDICTED MATCH SCORE: 54.13%
 Result: Potential Match - Review manually.


In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 98.7 MB/s eta 0:00:00


In [ ]:
import streamlit as st
import torch
import torch.nn as nn
import PyPDF2
import pickle
import re
import os
from nltk.corpus import stopwords

st.set_page_config(page_title="ATS Pro Analyzer", page_icon="📝", layout="wide")

st.markdown("""
    <style>
    .main { background-color: #ffffff; }
    .stTextArea textarea { background-color: #f8f9fa; }
    .report-card {
        padding: 20px;
        border-radius: 10px;
        border: 1px solid #e0e0e0;
        background-color: #ffffff;
        box-shadow: 2px 2px 5px rgba(0,0,0,0.05);
    }
    </style>
    """, unsafe_content_allowed=True)
def clean_text(text):
    stop_words = set(stopwords.words('english'))
    text = text.lower()
    text = re.sub(r"[\[\]'\".;:\|%()]", " ", text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = [word for word in text.split() if word not in stop_words]
    return " ".join(tokens)

def encode_input(text, vocab, max_len=256):
    tokens = text.split()
    encoded = [vocab.get(t, 0) for t in tokens[:max_len]]
    padding = [0] * (max_len - len(encoded))
    return encoded + padding

@st.cache_resource
def load_model_and_vocab():
    v_path = "checkpoints/efficient_vocab.pkl"
    m_path = "checkpoints/masked_siamese_model.pth"
    with open(v_path, "rb") as f:
        vocab = pickle.load(f)
    device = torch.device("cpu")
    model = MaskedSiameseTransformer(vocab_size=len(vocab))
    model.load_state_dict(torch.load(m_path, map_location=device))
    model.eval()
    return model, vocab

def main():
    st.title(" ATS Intelligence System")
    st.write("Upload your CV and paste the Job Description for a detailed AI analysis.")
    st.divider()
    try:
        model, vocab = load_model_and_vocab()
    except Exception as e:
        st.error(f"Error loading model: {e}. Please ensure checkpoints are present.")
        return
    col1, col2 = st.columns([1, 1], gap="large")
    with col1:
        st.subheader(" Upload Resume")
        uploaded_file = st.file_uploader("Upload PDF version only", type="pdf")
    with col2:
        st.subheader(" Job Description")
        jd_input = st.text_area("Paste the target JD here...", height=200, placeholder="Required skills, experience, etc.")
    if st.button("Analyze My CV", type="primary", use_container_width=True):
        if uploaded_file and jd_input:
            with st.spinner("Our Transformer model is analyzing your profile..."):
                pdf_reader = PyPDF2.PdfReader(uploaded_file)
                resume_raw = " ".join([page.extract_text() for page in pdf_reader.pages])
                res_clean = clean_text(resume_raw)
                jd_clean = clean_text(jd_input)
                res_seq = torch.tensor([encode_input(res_clean, vocab)])
                jd_seq = torch.tensor([encode_input(jd_clean, vocab)])
                res_mask = (res_seq != 0).float()
                jd_mask = (jd_seq != 0).float()
                with torch.no_grad():
                    logits = model(res_seq, jd_seq, res_mask, jd_mask)
                    score = torch.sigmoid(logits).item() * 100
                st.divider()
                st.header(f" Analysis Report: {score:.1f}% Match")
                st.progress(score / 100)
                with st.container():
                    st.markdown('<div class="report-card">', unsafe_content_allowed=True)
                    st.write("###  Core Strengths")
                    common_keywords = set(res_clean.split()) & set(jd_clean.split())
                    if common_keywords:
                        st.write(f"The model detected high relevance in these key areas: **{', '.join(list(common_keywords)[:10])}**")
                    else:
                        st.write("No direct keyword overlaps detected. The model is relying on semantic similarity.")
                    st.write("### Areas for Improvement")
                    missing_keywords = set(jd_clean.split()) - set(res_clean.split())
                    if missing_keywords:
                        st.write(f"Consider adding or emphasizing these missing requirements: **{', '.join(list(missing_keywords)[:8])}**")
                    st.write("### AI Conclusion")
                    if score > 75:
                        st.success("Your resume is a top-tier match for this position. The Transformer model indicates a very strong alignment of skills and experience.")
                    elif score > 50:
                        st.warning("You are a moderate match. While you have the core background, you may need to tailor your resume specifically to the missing JD keywords noted above.")
                    else:
                        st.error("Low match score. It is recommended to significantly update your resume to better reflect the technical stack mentioned in the JD.")
                    st.markdown('</div>', unsafe_content_allowed=True)
        else:
            st.error("Please provide both a Resume file and a Job Description.")

if __name__ == "__main__":
    main()

2026-03-28 06:24:17.031 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


TypeError: MarkdownMixin.markdown() got an unexpected keyword argument 'unsafe_content_allowed'

In [ ]:
# Step 1: Install LocalTunnel
!npm install -g localtunnel

# Step 2: Run Streamlit in the background
# Replace 'app.py' with whatever you named your script file
!streamlit run app.py &>/dev/null &

# Step 3: Expose the port to a public URL
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 3s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙your url is: https://neat-signs-hunt.loca.lt
